In [29]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random

In [30]:
# set seed for reproducibility
random.seed(42)
np.random.seed(42)

In [31]:
# generate synthetic data
n_listings = 500

# use real info
regions = ['Ile-de-France', 'Provence-Alpes', 'Nouvelle-Aquitaine', 
           'Auvergne-Rhone-Alpes', 'Occitanie', 'Hauts-de-France']
cities = {
    'Ile-de-France': ['Paris', 'Boulogne-Billancourt', 'Saint-Denis', 'Versailles'],
    'Provence-Alpes': ['Marseille', 'Nice', 'Toulon', 'Aix-en-Provence'],
    'Nouvelle-Aquitaine': ['Bordeaux', 'Limoges', 'Poitiers', 'Pau'],
    'Auvergne-Rhone-Alpes': ['Lyon', 'Grenoble', 'Clermont-Ferrand', 'Saint-Etienne'],
    'Occitanie': ['Toulouse', 'Montpellier', 'Nîmes', 'Perpignan'],
    'Hauts-de-France': ['Lille', 'Amiens', 'Roubaix', 'Dunkerque']
}

property_types = ['apartment', 'house', 'studio', 'loft', 'duplex']
agents = [f'agent_{i:03d}' for i in range(1, 21)]  # 20 agents

In [34]:
listings_data = []
start_date = datetime(2025, 1, 1)

In [35]:
for i in range(1, n_listings + 1):
    region = random.choice(regions)
    city = random.choice(cities[region])
    
    # base price depending on property type and city
    property_type = random.choice(property_types)
    if property_type == 'apartment':
        base_price = random.randint(200000, 800000)
    elif property_type == 'house':
        base_price = random.randint(350000, 1200000)
    elif property_type == 'studio':
        base_price = random.randint(100000, 250000)
    else:  # loft, duplex
        base_price = random.randint(300000, 900000)
    
    # price for Paris 
    if city == 'Paris': #capital
        base_price = int(base_price * 2.5)
    elif city in ['Lyon', 'Bordeaux', 'Nice']: #just big cities
        base_price = int(base_price * 1.5)
    
    created_at = start_date + timedelta(days=random.randint(0, 365))
    updated_at = created_at + timedelta(days=random.randint(0, 30)) if random.random() > 0.3 else created_at
    
    listing = {
        'listing_id': f'id_{i:04d}',
        'property_type': property_type,
        'city': city,
        'region': region,
        'price': base_price,
        'created_at': created_at.strftime('%Y-%m-%d'),
        'updated_at': updated_at.strftime('%Y-%m-%d'),
        'agent_id': random.choice(agents)
    }
    listings_data.append(listing)

df_listings = pd.DataFrame(listings_data)
df_listings.head(5)

,listing_id,property_type,city,region,price,created_at,updated_at,agent_id
0,id_0001,apartment,Montpellier,Occitanie,670852,2025-03-30,2025-04-04,agent_007
1,id_0002,apartment,Nice,Provence-Alpes,965910,2025-02-10,2025-02-23,agent_007
2,id_0003,duplex,Amiens,Hauts-de-France,570413,2025-06-13,2025-07-12,agent_003
3,id_0004,duplex,Versailles,Ile-de-France,716656,2025-10-10,2025-10-18,agent_017
4,id_0005,duplex,Lyon,Auvergne-Rhone-Alpes,1271917,2025-08-03,2025-08-03,agent_005


In [36]:
df_listings.describe()
df_listings.updated_at.max()

'2026-01-21'

In [37]:
#  listings data is ready 
df_listings.to_csv('synthetic_listings.csv', index=False)

In [44]:
# leads data
n_leads = 200

contact_sources = ['organic', 'paid', 'partner']

leads_data = []

for i in range(1, n_leads + 1):
    listing_id = random.choice(df_listings['listing_id'].values)
    listing_created = df_listings[df_listings['listing_id'] == listing_id]['created_at'].values[0]
    listing_created = datetime.strptime(listing_created, '%Y-%m-%d')
    
    # leads can arrive within 45 days after the listing is created
    max_days = 45
    contact_timestamp = listing_created + timedelta(
        days=random.randint(0, max_days),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59)
    )
    
    # Some leads may arrive with a delay (for testing late-arriving leads)
    if random.random() < 0.1:  # 10% of leads arrive with a significant delay
        contact_timestamp = listing_created + timedelta(days=random.randint(46, 60))
    
    lead = {
        'contact_id': f'id_{i:04d}',
        'listing_id': listing_id,
        'contact_source': random.choice(contact_sources),
        'contact_timestamp': contact_timestamp.strftime('%Y-%m-%d %H:%M:%S')
    }
    leads_data.append(lead)

df_leads = pd.DataFrame(leads_data)
df_leads.head(5)

,contact_id,listing_id,contact_source,contact_timestamp
0,id_0001,id_0181,partner,2026-01-16 16:04:00
1,id_0002,id_0248,organic,2025-07-11 07:28:00
2,id_0003,id_0292,paid,2025-06-23 17:15:00
3,id_0004,id_0346,paid,2025-03-25 21:08:00
4,id_0005,id_0175,partner,2025-05-28 03:05:00


In [45]:
df_leads.describe()

,contact_id,listing_id,contact_source,contact_timestamp
count,200,200,200,200
unique,200,170,3,198
top,id_0001,id_0175,partner,2026-02-10 00:00:00
freq,1,3,70,2


In [46]:
df_leads.contact_timestamp.max()

'2026-02-10 00:00:00'

In [47]:
#  leads data is ready 
df_leads.to_csv('synthetic_leads.csv', index=False)